In [8]:
!pip install unsloth

In [9]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

max_seq_length = 2048


In [10]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,   # QLoRA: 4-bit, fits comfortably on a free T4
)


==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # LoRA rank
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [12]:
dataset = load_dataset("b-mc2/sql-create-context", split = "train")
# columns: 'question', 'context' (the CREATE TABLE schema), 'answer' (the SQL)

# Use a subset so a training run takes minutes, not hours
dataset = dataset.shuffle(seed=42).select(range(3000))

def format_example(row):
    messages = [
        {"role": "system",
         "content": "You are a SQL expert. Given a database schema and a "
                    "question, output only the SQL query that answers it."},
        {"role": "user",
         "content": f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"},
        {"role": "assistant",
         "content": row["answer"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = dataset.map(format_example)

In [13]:
# Hold out a test set you NEVER train on — this is what makes your numbers credible
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, test_ds = split["train"], split["test"]

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        seed = 42,
        output_dir = "outputs",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2700 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 2 | Total steps = 676
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.008702
20,0.710696
30,0.678184
40,0.687033
50,0.645895
60,0.622419
70,0.636330
80,0.657801
90,0.635977
100,0.688754


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-676/tokenizer_config.json.


TrainOutput(global_step=676, training_loss=0.6058786740669837, metrics={'train_runtime': 1522.4691, 'train_samples_per_second': 3.547, 'train_steps_per_second': 0.444, 'total_flos': 9308172929531904.0, 'train_loss': 0.6058786740669837, 'epoch': 2.0})

In [14]:
FastLanguageModel.for_inference(model)

def generate_sql(schema, question):
    messages = [
        {"role": "system",
         "content": "You are a SQL expert. Given a database schema and a "
                    "question, output only the SQL query that answers it."},
        {"role": "user",
         "content": f"Schema:\n{schema}\n\nQuestion: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=128,
                         use_cache=True, temperature=0.0, do_sample=False)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return text.strip()

def normalize(sql):
    return " ".join(sql.lower().replace(";", "").replace('"', "'").split())

# Evaluate on the held-out test set
correct = 0
N = 200
for row in test_ds.select(range(N)):
    pred = generate_sql(row["context"], row["question"])
    if normalize(pred) == normalize(row["answer"]):
        correct += 1
print(f"Fine-tuned exact-match: {correct/N:.1%}")

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

Fine-tuned exact-match: 72.5%


In [ ]:
# Base Model

from unsloth import FastLanguageModel

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)

def normalize(sql):
    return " ".join(sql.lower().replace(";", "").replace('"', "'").split())

def generate_sql_base(schema, question):
    messages = [
        {"role": "system", "content": "You are a SQL expert. Given a database schema and a question, output only the SQL query that answers it."},
        {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion: {question}"},
    ]
    inputs = base_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = base_model.generate(input_ids=inputs, max_new_tokens=128,
                              use_cache=True, temperature=0.0, do_sample=False)
    return base_tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

dataset = load_dataset("b-mc2/sql-create-context", split = "train")
# columns: 'question', 'context' (the CREATE TABLE schema), 'answer' (the SQL)

# Use a subset so a training run takes minutes, not hours
dataset = dataset.shuffle(seed=42).select(range(3000))
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, test_ds = split["train"], split["test"]

def format_example(row):
    messages = [
        {"role": "system",
         "content": "You are a SQL expert. Given a database schema and a "
                    "question, output only the SQL query that answers it."},
        {"role": "user",
         "content": f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"},
        {"role": "assistant",
         "content": row["answer"]},
    ]
    text = base_tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

# Print a few first so you SEE why the base model fails
for row in test_ds.select(range(3)):
    print("Q:", row["question"])
    print("BASE:", generate_sql_base(row["context"], row["question"]))
    print("GOLD:", row["answer"], "\n")
    print(f"{normalize(generate_sql_base(row["context"], row["question"])) == normalize(row["answer"])}")

correct = 0
N = 200
for row in test_ds.select(range(N)):
    if normalize(generate_sql_base(row["context"], row["question"])) == normalize(row["answer"]):
        correct += 1
print(f"Base exact-match: {correct/N:.1%}")

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the number of the division for the 1st round?


...

BASE: SQL QUERY:
SELECT date FROM table_name_18 WHERE opponent = 'Lizards' AND result = 'w 18-17'
GOLD: SELECT date FROM table_name_18 WHERE opponent = "lizards" AND result = "w 18-17" 



Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


False


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Base exact-match: 41.5%
